In [30]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from sentence_transformers import SentenceTransformer


In [20]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('..', 'EDP')))

In [22]:
from parser_2 import Parser2

In [27]:
xml_file_path = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/output_solr34.xml'

raw_data = Parser2.XLMtoString(xml_file_path)

In [28]:
display(raw_data[:10])

only_text = [x['text'] for x in raw_data]

display(only_text[:10])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195',
  'text': 'A Fishmo

['Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660',
 'The Annunciation by Francesco Solimena, dated 1693 - 1693',
 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611',
 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579',
 'St Barbara by Parmigianino, dated None',
 "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737",
 'A Fishmonger at the Door by Jacob Ochtervelt, dated 1663 - 1663',
 'Portrait of a Deceased Girl, probably Catharina Margaretha van Valkenburg by Johannes Thopas, dated 1682 - 1682',
 'View of the Dam and the Damrak in Amsterdam by Jacob van Ruisdael, dated 1675 - 1672',
 'Hall Settle by Anonymous (Northern Netherlands), dated 1720 - 1700']

In [36]:
limit = 128

def chunker(contexts: list):
    chunks = []
    all_contexts = ' '.join(contexts).split('.')
    chunk = []
    for context in all_contexts:
        chunk.append(context)
        if len(chunk) >= 3 and len('.'.join(chunk)) > limit:
            # surpassed limit so add to chunks and reset
            chunks.append('.'.join(chunk).strip()+'.')
            # add some overlap between passages
            chunk = chunk[-2:]
    # if we finish and still have a chunk, add it
    if chunk is not None:
        chunks.append('.'.join(chunk))
    return chunks

chunks = chunker(only_text)
chunks

ids = []
for i in range(len(chunks)):
    ids.append(i)

chunked_data = []
for raw in raw_data:
    chunks = chunker([raw['text']])
    for i, chunk in enumerate(chunks):
        chunk_id = f'{raw["id"]}_{i}'
        chunked_data.append({'id': chunk_id, 'text': chunk})

chunked_data[:10]


[{'id': 'ID: /2021672/resource_document_mauritshuis_397_0',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340_0',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426_0',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432_0',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354_0',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181_0',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195_0',
  'te

In [80]:
# Generate embeddings using BGEM3 model
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
embeddings = model.encode([x['text'] for x in chunked_data])

In [85]:
display(embeddings[0])

type(embeddings[0])

array([ 0.00862603,  0.01286135, -0.03376907, -0.06245549,  0.038642  ,
        0.05342454, -0.03670572,  0.04166102, -0.08377837, -0.01893644,
        0.04251882,  0.01255127, -0.03283908,  0.05781702,  0.00289618,
        0.00688295,  0.04334607,  0.00053359, -0.01224214,  0.05585308,
       -0.04602025, -0.00439995,  0.02129052,  0.03010044, -0.01374634,
       -0.04415502, -0.00353212,  0.03320289, -0.04548392, -0.01821419,
       -0.02135395,  0.02513982,  0.01648747, -0.02361496,  0.03936541,
        0.01675215, -0.01877876,  0.00692662, -0.02258343,  0.00630685,
       -0.0791724 ,  0.01772372, -0.04769325, -0.06543085,  0.00158488,
        0.03766426, -0.0607101 , -0.01223136, -0.06645203,  0.00556119,
       -0.01105865,  0.00908041,  0.02290435,  0.03782685,  0.02983762,
        0.03477437, -0.02688894,  0.01732576, -0.04402058, -0.04607917,
       -0.02115682, -0.05822206, -0.06271311,  0.01808987,  0.0097702 ,
       -0.05360793, -0.0388708 , -0.01604171, -0.04952205,  0.03

numpy.ndarray

In [86]:
entities = [
    [item['id'] for item in chunked_data], #IDs
    [item['text'] for item in chunked_data], #Text
    embeddings #Embeddings
]

In [87]:
from pymilvus import MilvusClient
from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

# client = MilvusClient("milvus_demo.db")


In [88]:
connections.connect("default", host="localhost", port="19530")

In [89]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=100),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=512),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=512),  # Ensure the dimension matches your embeddings
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'sbert_demo'
col = Collection(col_name, schema, consistency_level="Strong")

In [90]:
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

In [305]:
# if col.has_collection(col_name):
#     print("Collection exists.")
    
# # # Create an index for the dense vector field
# # dense_index = client.prepare_index_params(index_type="IVF_FLAT", metric_type="L2", params={"nlist": 1024})

# # client.create_index(col_name, dense_index)

In [ ]:
# sparse_index = client.prepare_index_params(index_type="FLAT", metric_type="JACCARD", params={"nlist": 1024})
# client.create_index("sparse_vector", sparse_index)

In [91]:
col.insert(entities)

(insert count: 832, delete count: 0, upsert count: 0, timestamp: 451819928419041283, success count: 832, err count: 0, cost: 0)

In [92]:
col.flush()

In [105]:
query = "Vermeer"
query_embeddings = model.encode(query)
k=10

# display(query_embeddings)

# print(query_embeddings)

search_params = {"metric_type": "IP"}

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.search(
    data = [query_embeddings],
    anns_field="dense_vector",
    param=search_params,
    limit=10,
    output_fields=["pk","text"]
)

for result in res[0]:
    print(result)

id: ID: /2021672/resource_document_mauritshuis_92_0, distance: 0.2322172075510025, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_92_0', 'text': 'View of Delft by Johannes Vermeer, dated 1661 - 1660'}
id: ID: /2021672/resource_document_mauritshuis_406_0, distance: 0.21027331054210663, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_406_0', 'text': 'Diana and her Nymphs by Johannes Vermeer, dated 1654 - 1653'}
id: ID: /2021672/resource_document_mauritshuis_59_0, distance: 0.20405493676662445, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_59_0', 'text': "The Raven Robbed of the Feathers He Wore to Adorn Himself by Melchior d' Hondecoeter, dated 1671 - 1671"}
id: ID: /2021672/resource_document_mauritshuis_670_0, distance: 0.19617575407028198, entity: {'pk': 'ID: /2021672/resource_document_mauritshuis_670_0', 'text': 'Girl with a Pearl Earring by Johannes Vermeer, dated 1665 - 1665'}
